In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
columns = [
    "checking_status", "duration", "credit_history", "purpose", "credit_amount",
    "savings_status", "employment", "installment_commitment", "personal_status",
    "other_parties", "residence_since", "property_magnitude", "age",
    "other_payment_plans", "housing", "existing_credits", "job",
    "num_dependents", "own_telephone", "foreign_worker", "class"
]

df = pd.read_csv(url, sep=" ", header=None, names=columns)
print(df.shape)


(1000, 21)


In [2]:
df['bad_risk'] = df['class'].map({1: 0, 2: 1})

df['monthly_repayment'] = df['credit_amount'] / df['duration']
df['log_credit_amount'] = np.log(df['credit_amount'])
df['age_band'] = pd.cut(df['age'], bins=[18, 25, 40, 60, 100],
                        labels=['young', 'adult', 'middle_aged', 'senior'])

In [3]:
from sklearn.model_selection import train_test_split
y = df['bad_risk']
X = df.drop(columns=['class', 'bad_risk'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(800, 23) (200, 23)
0.3 0.3


In [4]:

X_train.columns.tolist()

['checking_status',
 'duration',
 'credit_history',
 'purpose',
 'credit_amount',
 'savings_status',
 'employment',
 'installment_commitment',
 'personal_status',
 'other_parties',
 'residence_since',
 'property_magnitude',
 'age',
 'other_payment_plans',
 'housing',
 'existing_credits',
 'job',
 'num_dependents',
 'own_telephone',
 'foreign_worker',
 'monthly_repayment',
 'log_credit_amount',
 'age_band']

In [5]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
print(cv.get_n_splits())

15


In [6]:
train = X_train.copy()
train['bad_risk'] = y_train
train.groupby('checking_status')['bad_risk'].mean()

,bad_risk
checking_status,
A11,0.479452
A12,0.392523
A13,0.254902
A14,0.120253


In [7]:
train.groupby('savings_status')['bad_risk'].agg(['mean', 'count']).sort_values('mean')

,mean,count
savings_status,,
A64,0.121951,41
A63,0.160000,50
A65,0.166667,144
A62,0.333333,90
A61,0.364211,475


In [8]:
train.groupby('employment')['bad_risk'].agg(['mean', 'count']).sort_values('mean')

,mean,count
employment,,
A74,0.223022,139
A75,0.262136,206
A73,0.276753,271
A71,0.414634,41
A72,0.440559,143


In [9]:
train.groupby('credit_history')['bad_risk'].agg(['mean', 'count']).sort_values('mean')
#moving to one-hot .

,mean,count
credit_history,,
A34,0.175000,240
A33,0.319444,72
A32,0.322115,416
A31,0.536585,41
A30,0.612903,31


In [10]:
train.groupby('job')['bad_risk'].agg(['mean', 'count']).sort_values('mean')
# moving to one-hot encode.

,mean,count
job,,
A173,0.289628,511
A172,0.291139,158
A174,0.347826,115
A171,0.375000,16


In [11]:
ordinal_cols = ['checking_status', 'savings_status', 'employment', 'age_band']
onehot_cols  = ['purpose', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'own_telephone', 'foreign_worker','credit_history','job']   # 10 columns, fill in from the table above
numeric_cols = [c for c in X_train.columns if c not in ordinal_cols + onehot_cols]

print(len(ordinal_cols), len(onehot_cols), len(numeric_cols))
print(numeric_cols)

4 10 9
['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents', 'monthly_repayment', 'log_credit_amount']


In [12]:
checking_order   = ['A14', 'A13', 'A12', 'A11']
savings_order    = ['A64', 'A63', 'A65', 'A62', 'A61']
employment_order = ['A75', 'A74', 'A73', 'A72', 'A71']
age_order        = ['senior', 'middle_aged', 'adult', 'young']

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(
            categories=[checking_order, savings_order, employment_order, age_order],
            handle_unknown='use_encoded_value', unknown_value=-1),
         ordinal_cols),
        ('ohe', OneHotEncoder(handle_unknown='ignore'), onehot_cols),
    ],
    remainder='passthrough'   # numeric columns pass through unchanged
)

In [14]:
X_check = preprocessor.fit_transform(X_train)
print(X_check.shape)

(800, 53)


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', DummyClassifier(strategy='most_frequent'))
])

acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')

print(len(acc), acc.mean(), acc.std())
print(auc.mean())

15 0.6999999999999998 1.1102230246251565e-16
0.5


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

preprocessor_lr = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(
            categories=[checking_order, savings_order, employment_order, age_order],
            handle_unknown='use_encoded_value', unknown_value=-1),
         ordinal_cols),
        ('ohe', OneHotEncoder(handle_unknown='ignore'), onehot_cols),
        ('num', StandardScaler(), numeric_cols),
    ]
)

In [17]:
lr_pipe = Pipeline([
    ('prep', preprocessor_lr),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
])

auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=cv, scoring='roc_auc')
print(len(auc_lr), auc_lr.mean(), auc_lr.std())

15 0.7786830357142857 0.04325407846730351


In [18]:
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=1000,
        max_depth=10,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=42))
])

auc_rf = cross_val_score(rf_pipe, X_train, y_train, cv=cv, scoring='roc_auc')
print(len(auc_rf), auc_rf.mean(), auc_rf.std())

15 0.7914682539682539 0.034330862058035976


In [19]:
from sklearn.model_selection import cross_validate

res = cross_validate(rf_pipe, X_train, y_train, cv=cv,
                     scoring='roc_auc', return_train_score=True)
print(res['train_score'].mean(), res['test_score'].mean())

0.9287830171130954 0.7914682539682539


In [20]:
rf_small = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=20,
        class_weight='balanced', random_state=42))
])

res_small = cross_validate(rf_small, X_train, y_train, cv=cv,
                           scoring='roc_auc', return_train_score=True)
print(res_small['train_score'].mean(), res_small['test_score'].mean())

0.8607390873015873 0.7818824404761905


In [21]:
res_lr = cross_validate(lr_pipe, X_train, y_train, cv=cv,
                        scoring='roc_auc', return_train_score=True)
print(res_lr['train_score'].mean(), res_lr['test_score'].mean())

0.8357414124503967 0.7786830357142857


In [22]:
from xgboost import XGBClassifier

xgb_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', XGBClassifier(
        n_estimators=200, max_depth=2, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=2.33,        # 70 good / 30 bad
        eval_metric='logloss', random_state=42))
])

res_xgb = cross_validate(xgb_pipe, X_train, y_train, cv=cv,
                         scoring='roc_auc', return_train_score=True)
print(res_xgb['train_score'].mean(), res_xgb['test_score'].mean())

0.913201419890873 0.7933035714285716


In [23]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba = cross_val_predict(lr_pipe, X_train, y_train, cv=skf,
                          method='predict_proba')[:, 1]

pred = (proba >= 0.5).astype(int)
print(confusion_matrix(y_train, pred))

[[397 163]
 [ 75 165]]


In [24]:
import numpy as np

for t in np.arange(0.1, 0.9, 0.05):
    pred_t = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred_t).ravel()
    cost = 5 * fn + 1 * fp
    print(round(t, 2), "missed bad:", fn, "wrong rejects:", fp, "cost:", cost)

0.1 missed bad: 5 wrong rejects: 453 cost: 478
0.15 missed bad: 11 wrong rejects: 406 cost: 461
0.2 missed bad: 15 wrong rejects: 360 cost: 435
0.25 missed bad: 26 wrong rejects: 316 cost: 446
0.3 missed bad: 31 wrong rejects: 270 cost: 425
0.35 missed bad: 42 wrong rejects: 243 cost: 453
0.4 missed bad: 52 wrong rejects: 214 cost: 474
0.45 missed bad: 67 wrong rejects: 186 cost: 521
0.5 missed bad: 75 wrong rejects: 163 cost: 538
0.55 missed bad: 86 wrong rejects: 137 cost: 567
0.6 missed bad: 98 wrong rejects: 110 cost: 600
0.65 missed bad: 115 wrong rejects: 90 cost: 665
0.7 missed bad: 130 wrong rejects: 76 cost: 726
0.75 missed bad: 147 wrong rejects: 55 cost: 790
0.8 missed bad: 171 wrong rejects: 34 cost: 889
0.85 missed bad: 186 wrong rejects: 25 cost: 955


In [25]:
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, average_precision_score

pred = (proba >= 0.3).astype(int)
print(confusion_matrix(y_train, pred))
print("recall   ", recall_score(y_train, pred))
print("precision", precision_score(y_train, pred))
print("f1       ", f1_score(y_train, pred))
print("roc_auc  ", roc_auc_score(y_train, proba))
print("pr_auc   ", average_precision_score(y_train, proba))

[[290 270]
 [ 31 209]]
recall    0.8708333333333333
precision 0.4363256784968685
f1        0.5813630041724618
roc_auc   0.7725892857142858
pr_auc    0.5867356291550009


In [26]:
lr_pipe.fit(X_train, y_train)
proba_test = lr_pipe.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.3).astype(int)

print(confusion_matrix(y_test, pred_test))
print("recall   ", recall_score(y_test, pred_test))
print("precision", precision_score(y_test, pred_test))
print("f1       ", f1_score(y_test, pred_test))
print("roc_auc  ", roc_auc_score(y_test, proba_test))
print("pr_auc   ", average_precision_score(y_test, proba_test))

[[71 69]
 [ 9 51]]
recall    0.85
precision 0.425
f1        0.5666666666666667
roc_auc   0.8129761904761905
pr_auc    0.6607070653554062


In [27]:
import numpy as np
rng = np.random.default_rng(42)
y_arr = y_test.to_numpy()

aucs = []
for _ in range(1000):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    if y_arr[idx].min() == y_arr[idx].max():
        continue
    aucs.append(roc_auc_score(y_arr[idx], proba_test[idx]))

print(np.percentile(aucs, [2.5, 50, 97.5]))

[0.74121379 0.81381557 0.87955556]


In [28]:
import pandas as pd

names = lr_pipe.named_steps['prep'].get_feature_names_out()
coefs = lr_pipe.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({'feature': names, 'coef': coefs})
coef_df['abs'] = coef_df['coef'].abs()
print(coef_df.sort_values('abs', ascending=False).head(10))

                         feature      coef       abs
11              ohe__purpose_A46  0.948145  0.948145
39       ohe__credit_history_A34 -0.826595  0.826595
5               ohe__purpose_A41 -0.809972  0.809972
35       ohe__credit_history_A30  0.675649  0.675649
24  ohe__property_magnitude_A124  0.650079  0.650079
34      ohe__foreign_worker_A202 -0.639722  0.639722
33      ohe__foreign_worker_A201  0.635009  0.635009
6              ohe__purpose_A410 -0.616467  0.616467
10              ohe__purpose_A45  0.590634  0.590634
0           ord__checking_status  0.531129  0.531129


In [29]:
prep = lr_pipe.named_steps['prep']
Xtr = prep.transform(X_train)
Xte = prep.transform(X_test)
if hasattr(Xtr, 'toarray'):
    Xtr, Xte = Xtr.toarray(), Xte.toarray()

i = np.where(pred_test == 1)[0][0]          # the first rejected applicant in the test set
contrib = coefs * (Xte[i] - Xtr.mean(axis=0))

why = pd.DataFrame({'feature': names, 'push': contrib}).sort_values('push')

print(X_test.iloc[i])
print("chance of bad:", proba_test[i], " actual:", y_test.iloc[i])
print(why.tail(5))    # the reasons pushing toward reject
print(why.head(5))    # the reasons pulling toward approve

checking_status                  A12
duration                          18
credit_history                   A32
purpose                          A49
credit_amount                   1913
savings_status                   A64
employment                       A72
installment_commitment             3
personal_status                  A94
other_parties                   A101
residence_since                    3
property_magnitude              A121
age                               36
other_payment_plans             A141
housing                         A152
existing_credits                   1
job                             A173
num_dependents                     1
own_telephone                   A192
foreign_worker                  A201
monthly_repayment         106.277778
log_credit_amount           7.556428
age_band                       adult
Name: 30, dtype: object
chance of bad: 0.33796748845534785  actual: 0
                          feature      push
39        ohe__credit_history_A34  

In [30]:
X_train['personal_status'].value_counts()

,count
personal_status,
A93,430
A92,249
A94,80
A91,41


In [31]:
fair = X_train.copy()
fair['sex'] = np.where(fair['personal_status'] == 'A92', 'female', 'male')
fair['bad'] = y_train.values
fair['reject'] = (proba >= 0.3).astype(int)

def summary(g):
    return pd.Series({
        'n': len(g),
        'reject_rate': g['reject'].mean(),
        'bad_caught': g.loc[g['bad'] == 1, 'reject'].mean(),
        'good_wrongly_rejected': g.loc[g['bad'] == 0, 'reject'].mean(),
    })

print(fair.groupby('sex').apply(summary))
print(fair.groupby('age_band', observed=True).apply(summary))

            n  reject_rate  bad_caught  good_wrongly_rejected
sex                                                          
female  249.0     0.678715    0.887640                 0.5625
male    551.0     0.562613    0.860927                 0.4500
                 n  reject_rate  bad_caught  good_wrongly_rejected
age_band                                                          
young        146.0     0.794521    0.916667               0.709302
adult        444.0     0.565315    0.842975               0.461300
middle_aged  178.0     0.544944    0.900000               0.406250
senior        32.0     0.468750    0.777778               0.347826


/tmp/ipykernel_3213/946215869.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(fair.groupby('sex').apply(summary))
/tmp/ipykernel_3213/946215869.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(fair.groupby('age_band', observed=True).apply(summary))


In [32]:
onehot2 = [c for c in onehot_cols if c != 'personal_status']

prep2 = ColumnTransformer(transformers=[
    ('ord', OrdinalEncoder(
        categories=[checking_order, savings_order, employment_order, age_order],
        handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
    ('ohe', OneHotEncoder(handle_unknown='ignore'), onehot2),
    ('num', StandardScaler(), numeric_cols),
])
lr2 = Pipeline([('prep', prep2),
                ('model', LogisticRegression(class_weight='balanced', max_iter=1000))])

proba2 = cross_val_predict(lr2, X_train, y_train, cv=skf, method='predict_proba')[:, 1]
fair['reject2'] = (proba2 >= 0.3).astype(int)

print("AUC without personal_status:", roc_auc_score(y_train, proba2), "(before: 0.7726)")
print(fair.groupby('sex')['reject2'].mean())
print("actual bad rate by sex:\n", fair.groupby('sex')['bad'].mean())
print("actual bad rate by age band:\n", fair.groupby('age_band', observed=True)['bad'].mean())

AUC without personal_status: 0.767998511904762 (before: 0.7726)
sex
female    0.610442
male      0.604356
Name: reject2, dtype: float64
actual bad rate by sex:
 sex
female    0.357430
male      0.274047
Name: bad, dtype: float64
actual bad rate by age band:
 age_band
young          0.410959
adult          0.272523
middle_aged    0.280899
senior         0.281250
Name: bad, dtype: float64


In [33]:
def summary2(g):
    return pd.Series({
        'n': len(g),
        'reject_rate': g['reject2'].mean(),
        'bad_caught': g.loc[g['bad'] == 1, 'reject2'].mean(),
        'good_wrongly_rejected': g.loc[g['bad'] == 0, 'reject2'].mean(),
    })

print(fair.groupby('sex')[['reject2', 'bad']].apply(summary2))

            n  reject_rate  bad_caught  good_wrongly_rejected
sex                                                          
female  249.0     0.610442    0.865169                0.46875
male    551.0     0.604356    0.874172                0.50250


In [34]:
for t in np.arange(0.2, 0.45, 0.05):
    p = (proba2 >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, p).ravel()
    print(round(t, 2), "missed bad:", fn, "wrong rejects:", fp, "cost:", 5 * fn + fp)

0.2 missed bad: 18 wrong rejects: 363 cost: 453
0.25 missed bad: 25 wrong rejects: 314 cost: 439
0.3 missed bad: 31 wrong rejects: 276 cost: 431
0.35 missed bad: 41 wrong rejects: 245 cost: 450
0.4 missed bad: 51 wrong rejects: 222 cost: 477


In [35]:
lr2.fit(X_train, y_train)
proba_test2 = lr2.predict_proba(X_test)[:, 1]
pred_test2 = (proba_test2 >= 0.3).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test2).ravel()
print([[tn, fp], [fn, tp]])
print("recall   ", recall_score(y_test, pred_test2))
print("precision", precision_score(y_test, pred_test2))
print("roc_auc  ", roc_auc_score(y_test, proba_test2))
print("cost     ", 5 * fn + fp)

[[np.int64(71), np.int64(69)], [np.int64(8), np.int64(52)]]
recall    0.8666666666666667
precision 0.4297520661157025
roc_auc   0.8194047619047619
cost      109


In [36]:
import joblib
joblib.dump(lr2, 'credit_risk_pipeline.joblib')

['credit_risk_pipeline.joblib']